In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
!pip install gliner chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

# 🧠 PHẦN 1: XÂY DỰNG CƠ SỞ DỮ LIỆU TRI THỨC (`build_brain_db.py`)

Phần này thực hiện việc chuyển đổi các file tri thức phân cấp (Ontology) thành một cơ sở dữ liệu Vector có thể tìm kiếm được.

### 1.1. Cấu trúc và Khởi tạo
- **Persistent Storage**: Sử dụng `chromadb.PersistentClient(path=DB_PATH)` để lưu dữ liệu trực tiếp vào Google Drive, tránh mất dữ liệu khi kết thúc session.
- **Embedding Model**: Sử dụng `all-MiniLM-L6-v2` (384 chiều) thông qua `SentenceTransformerEmbeddingFunction` để chuyển văn bản thành vector.

### 1.2. Logic Xử lý Dữ liệu
- **`parse_ontology_recursive`**: Hàm đệ quy này là trọng tâm để xử lý các file JSON phức tạp. Nó duyệt qua từng node, tự động nối chuỗi `current_path` (ví dụ: `Software.Backend.Java`) để tạo metadata cho các kỹ năng lá.
- **`build_collection`**:
    - Gọi hàm đệ quy để thu thập `texts`, `metadatas`, và `ids`.
    - Sử dụng `get_or_create_collection` để quản lý các nhóm tri thức riêng biệt (`tech_ontology` và `soft_skills_ontology`).
    - Đẩy dữ liệu vào ChromaDB bằng lệnh `collection.add()`.

In [ ]:
import json
import os
import chromadb
from chromadb.utils import embedding_functions

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Đường dẫn tới các file JSON chứa cây tri thức (Ontology) trên Google Drive
TECH_ONTOLOGY_PATH = '/content/drive/MyDrive/Nlp resume ranker/taxonomy/taxonomy_processed/tech_ontology.json'
SOFT_ONTOLOGY_PATH = '/content/drive/MyDrive/Nlp resume ranker/taxonomy/taxonomy_processed/soft_skills_ontology.json'

# Thư mục lưu trữ cơ sở dữ liệu Vector (ChromaDB) trên Google Drive
DB_PATH = "/content/drive/MyDrive/Nlp resume ranker/chroma_db"

# Khởi tạo Client lưu trữ dữ liệu xuống ổ cứng (Persistent)
client = chromadb.PersistentClient(path=DB_PATH)

# Sử dụng mô hình Sentence-Transformer để chuyển văn bản thành Vector (384 chiều)
s_bert_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

def parse_ontology_recursive(node, current_path, texts, metadatas, ids):
    """
    Hàm đệ quy để duyệt toàn bộ cây JSON không giới hạn cấp độ.
    - node: Node hiện tại (dict, list hoặc str)
    - current_path: Chuỗi lưu vết vị trí (VD: 'Software.Backend.Java')
    """
    if isinstance(node, dict):
        for key, value in node.items():
            new_path = f"{current_path}.{key}" if current_path else key
            parse_ontology_recursive(value, new_path, texts, metadatas, ids)
    elif isinstance(node, list):
        for item in node:
            parse_ontology_recursive(item, current_path, texts, metadatas, ids)
    elif isinstance(node, str):
        cleaned_text = node.strip().lower()
        if cleaned_text and cleaned_text not in texts:
            # [DEBUG_LOG] Theo dõi quá trình bóc tách từng node lá
            # print(f"[DEBUG_LOG] Found Leaf: {cleaned_text} at {current_path}")
            texts.append(cleaned_text)
            metadatas.append({"path": current_path})
            ids.append(f"{current_path}_{cleaned_text}")

def build_collection(collection_name, file_path):
    """
    Hàm đọc file JSON, xử lý đệ quy và đẩy dữ liệu vào ChromaDB
    """
    print(f"\n{'='*20} BẮT ĐẦU XỬ LÝ: {collection_name} {'='*20}")
    if not os.path.exists(file_path):
        print(f"[!] Lỗi: Không tìm thấy file tại {file_path}. Hãy kiểm tra lại Mount Drive.")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    texts, metas, ids = [], [], []
    print(f"[DEBUG_LOG] Đang phân tích cấu trúc cây JSON...")
    parse_ontology_recursive(data, "", texts, metas, ids)
    print(f"[DEBUG_LOG] Đã bóc tách xong {len(texts)} thực thể.")

    print(f"[DEBUG_LOG] Đang khởi tạo/kết nối Collection trong ChromaDB...")
    collection = client.get_or_create_collection(name=collection_name, embedding_function=s_bert_ef)

    print(f"[DEBUG_LOG] Đang tiến hành Embedding và lưu xuống ổ cứng (có thể mất vài phút)... ")
    collection.add(documents=texts, metadatas=metas, ids=ids)
    print(f"[DEBUG_LOG] HOÀN TẤT: {collection_name} đã sẵn sàng tại {DB_PATH}")

if __name__ == "__main__":
    # Xây dựng não trái (Kỹ thuật)
    build_collection("tech_ontology", TECH_ONTOLOGY_PATH)
    # Xây dựng não phải (Kỹ năng mềm)
    build_collection("soft_skills_ontology", SOFT_ONTOLOGY_PATH)
    print("\n[DEBUG_LOG] --- TẤT CẢ NÃO BỘ ĐÃ ĐƯỢC XÂY DỰNG XONG ---")


==================== BẮT ĐẦU XỬ LÝ: tech_ontology ====================
[DEBUG_LOG] Đang phân tích cấu trúc cây JSON...
[DEBUG_LOG] Đã bóc tách xong 534 thực thể.
[DEBUG_LOG] Đang khởi tạo/kết nối Collection trong ChromaDB...
[DEBUG_LOG] Đang tiến hành Embedding và lưu xuống ổ cứng (có thể mất vài phút)... 
[DEBUG_LOG] HOÀN TẤT: tech_ontology đã sẵn sàng tại /content/drive/MyDrive/Nlp resume ranker/chroma_db

==================== BẮT ĐẦU XỬ LÝ: soft_skills_ontology ====================
[DEBUG_LOG] Đang phân tích cấu trúc cây JSON...
[DEBUG_LOG] Đã bóc tách xong 1814 thực thể.
[DEBUG_LOG] Đang khởi tạo/kết nối Collection trong ChromaDB...
[DEBUG_LOG] Đang tiến hành Embedding và lưu xuống ổ cứng (có thể mất vài phút)... 
[DEBUG_LOG] HOÀN TẤT: soft_skills_ontology đã sẵn sàng tại /content/drive/MyDrive/Nlp resume ranker/chroma_db

[DEBUG_LOG] --- TẤT CẢ NÃO BỘ ĐÃ ĐƯỢC XÂY DỰNG XONG ---


# 🚀 PHẦN 2: ENRICHMENT PIPELINE (`process_jd.py`)

Đây là hệ thống xử lý thời gian thực (Online) để chuẩn hóa và làm giàu dữ liệu từ các văn bản JD thô.

### 2.1. Thành phần Hệ thống
- **GLiNER (Zero-shot NER)**: Sử dụng model `gliner_multi-v2.1` với các nhãn tùy chỉnh như `Programming Language`, `IT Tool`, `Certification`, v.v. Điều này cho phép trích xuất linh hoạt mà không cần training lại.
- **Knowledge Retrieval**: Kết nối với 2 collections (`tech_col`, `soft_col`) đã build ở Phần 1.

### 2.2. Quy trình Xử lý `process_single_jd`
1.  **Trích xuất Ràng buộc (Regex)**: Tìm kiếm số năm kinh nghiệm bằng biểu thức chính quy (`\d+ year/năm`).
2.  **Trích xuất Thực thể (NER)**: GLiNER nhận diện các từ khóa trong văn bản.
3.  **Chuẩn hóa & Mapping (ChromaDB Query)**:
    - Với mỗi thực thể, thực hiện truy vấn vector với `n_results=1`.
    - Nếu `distance < 0.4` (ngưỡng tương đồng cao), lấy tên chuẩn từ database.
    - Nếu `distance > 0.4`, giữ nguyên tên thô từ JD.
4.  **Mở rộng Kỹ năng (Expansion)**:
    - Sử dụng `path` trong metadata để tìm các kỹ năng "anh em" (Siblings) trong cây tri thức gốc thông qua hàm `get_by_path`.
5.  **Hậu xử lý**:
    - Loại bỏ các kỹ năng đã có trong `Must_Have` khỏi danh sách `Nice_To_Have_Expansions` để tránh trùng lặp.
    - Chuyển đổi toàn bộ `set()` sang `list()` để lưu thành JSON hợp lệ.

In [ ]:
import re
import json
import os
import chromadb
from gliner import GLiNER
from pathlib import Path
from datetime import datetime

# --- KHỞI TẠO HỆ THỐNG ---
print(f"[DEBUG_LOG] Đang tải mô hình và kết nối database...")
DB_PATH = "/content/drive/MyDrive/Nlp resume ranker/chroma_db"
client = chromadb.PersistentClient(path=DB_PATH)

tech_col = client.get_collection(name="tech_ontology")
soft_col = client.get_collection(name="soft_skills_ontology")

# Sử dụng model đa ngôn ngữ v2.1
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

JD_INPUT_DIR = "/content/drive/MyDrive/Nlp resume ranker/JD"
JD_OUTPUT_DIR = "/content/drive/MyDrive/Nlp resume ranker/Cleaned_JD_V2"
TECH_JSON_PATH = '/content/drive/MyDrive/Nlp resume ranker/taxonomy/taxonomy_processed/tech_ontology.json'

os.makedirs(JD_OUTPUT_DIR, exist_ok=True)

def load_raw_ontology(path):
    try:
        with open(path, 'r', encoding='utf-8') as f: return json.load(f)
    except: return {}

def get_by_path(root, path_str):
    keys = path_str.split('.')
    for k in keys:
        if isinstance(root, dict): root = root.get(k, {})
    return root if isinstance(root, list) else []

def process_single_jd(jd_text, distance_threshold=0.4):
    # Khởi tạo cấu trúc output chuẩn hóa
    results = {
        "Metadata": {
            "Job_Title": "Unknown",
            "Processed_At": datetime.now().isoformat()
        },
        "Hard_Constraints": {
            "Min_Experience_Years": None,
            "Required_Degree": "Not specified"
        },
        "Hard_Skills": {
            "Must_Have": {
                "Tech_Skills": set(), # Gộp: Technical Skill, Methodology, Tool, Domain Knowledge, Language
                "Certifications": set() # Riêng: Certification
            },
            "Nice_To_Have_Expansions": {
                "Tech_Siblings": set()
            }
        },
        "Soft_Skills": set() # Nhãn: Soft Skill
    }

    # 1. TRÍCH XUẤT RÀNG BUỘC CỨNG (Regex)
    exp_match = re.search(r'(\d+)\s*(?:year|năm)', jd_text, re.IGNORECASE)
    results["Hard_Constraints"]["Min_Experience_Years"] = int(exp_match.group(1)) if exp_match else 0

    # 2. TRÍCH XUẤT THỰC THỂ THEO TẬP NHÃN CHUẨN ĐA MIỀN
    labels = ["Job Role", "Technical Skill", "Methodology", "Tool", "Certification", "Domain Knowledge", "Soft Skill", "Language", "Education"]
    entities = gliner_model.predict_entities(jd_text, labels)
    raw_tech_tree = load_raw_ontology(TECH_JSON_PATH)

    for ent in entities:
        label, text = ent['label'], ent['text'].strip()

        # Phân loại Metadata
        if label == "Job Role":
            results["Metadata"]["Job_Title"] = text

        # Phân loại Education
        elif label == "Education":
            results["Hard_Constraints"]["Required_Degree"] = text

        # GOM NHÓM VÀO HARD_SKILLS
        elif label in ["Technical Skill", "Methodology", "Tool", "Certification", "Domain Knowledge", "Language"]:
            query_res = tech_col.query(query_texts=[text.lower()], n_results=1)
            dist = query_res['distances'][0][0] if query_res['distances'] else 1.0

            # Quy tắc phân rổ trong Hard_Skills
            target_bucket = "Certifications" if label == "Certification" else "Tech_Skills"

            if dist < distance_threshold:
                matched_skill = query_res['documents'][0][0]
                path = query_res['metadatas'][0][0]['path']
                results["Hard_Skills"]["Must_Have"][target_bucket].add(matched_skill)

                # Mở rộng (Expansion) cho nhóm kỹ thuật/cứng
                if target_bucket == "Tech_Skills":
                    neighbors = get_by_path(raw_tech_tree, path)
                    if isinstance(neighbors, list):
                        for n in neighbors:
                            if isinstance(n, str): results["Hard_Skills"]["Nice_To_Have_Expansions"]["Tech_Siblings"].add(n.lower())
            else:
                # Nếu không tìm thấy trong DB, giữ nguyên text thô từ JD
                results["Hard_Skills"]["Must_Have"][target_bucket].add(text)

        # Phân loại Soft Skills
        elif label == "Soft Skill":
            query_res = soft_col.query(query_texts=[text.lower()], n_results=1)
            dist = query_res['distances'][0][0] if query_res['distances'] else 1.0
            final_soft = query_res['documents'][0][0] if dist < distance_threshold else text
            results["Soft_Skills"].add(final_soft)

    # 3. HẬU XỬ LÝ (Deduplication & List Conversion)
    must_have_tech = results["Hard_Skills"]["Must_Have"]["Tech_Skills"]
    results["Hard_Skills"]["Nice_To_Have_Expansions"]["Tech_Siblings"] = list(results["Hard_Skills"]["Nice_To_Have_Expansions"]["Tech_Siblings"].difference(must_have_tech))

    results["Hard_Skills"]["Must_Have"]["Tech_Skills"] = list(must_have_tech)
    results["Hard_Skills"]["Must_Have"]["Certifications"] = list(results["Hard_Skills"]["Must_Have"]["Certifications"])
    results["Soft_Skills"] = list(results["Soft_Skills"])

    return results

def batch_process_jds():
    print(f"\n{'='*20} BẮT ĐẦU XỬ LÝ HÀNG LOẠT {'='*20}")
    jd_files = [f for f in os.listdir(JD_INPUT_DIR) if os.path.isfile(os.path.join(JD_INPUT_DIR, f))]
    if not jd_files: return

    for filename in jd_files:
        try:
            with open(os.path.join(JD_INPUT_DIR, filename), 'r', encoding='utf-8') as f:
                jd_content = f.read()

            enriched_data = process_single_jd(jd_content)
            output_path = os.path.join(JD_OUTPUT_DIR, Path(filename).stem + "_cleaned.json")

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(enriched_data, f, indent=4, ensure_ascii=False)
            print(f"  [OK] Processed: {filename}")
        except Exception as e:
            print(f"  [ERROR] {filename}: {str(e)}")

if __name__ == "__main__":
    batch_process_jds()

[DEBUG_LOG] Đang tải mô hình và kết nối database...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


==================== BẮT ĐẦU XỬ LÝ HÀNG LOẠT ====================


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1037 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  [OK] Processed: JD_BA_1.txt


...

  [OK] Processed: JD_PM_1.txt


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 677 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  [OK] Processed: JD_SM_1.txt


/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 441 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


  [OK] Processed: JD_SW_4.txt


## 📊 KIỂM TRA KẾT QUẢ ĐẦU RA (process_jd.py)


In [ ]:
import os
import json

# Đường dẫn tới thư mục chứa JD đã xử lý
CLEANED_DIR = "/content/drive/MyDrive/Nlp resume ranker/Cleaned_JD_V2"

if not os.path.exists(CLEANED_DIR):
    print(f"[!] Thư mục {CLEANED_DIR} không tồn tại.")
else:
    cleaned_files = [f for f in os.listdir(CLEANED_DIR) if f.endswith('.json')]

    if not cleaned_files:
        print("[!] Không tìm thấy file JSON nào trong thư mục đầu ra.")
    else:
        print(f"[INFO] Đang hiển thị nội dung của {len(cleaned_files)} file đã xử lý theo SCHEMA MỚI:\n")

        for i, filename in enumerate(cleaned_files, 1):
            file_path = os.path.join(CLEANED_DIR, filename)

            print(f"{'#'*60}")
            print(f"### {i}. FILE: {filename}")
            print(f"{'#'*60}")

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    # In định dạng JSON đẹp mắt
                    print(json.dumps(data, indent=4, ensure_ascii=False))
            except Exception as e:
                print(f"[ERROR] Không thể đọc file {filename}: {e}")

            print("\n")

[INFO] Đang hiển thị nội dung của 17 file đã xử lý theo SCHEMA MỚI:

############################################################
### 1. FILE: JD_BA_1_cleaned.json
############################################################
{
    "Metadata": {
        "Job_Title": "Business Analyst",
        "Processed_At": "2026-04-12T00:26:27.551731"
    },
    "Hard_Constraints": {
        "Min_Experience_Years": 0,
        "Required_Degree": "Not specified"
    },
    "Hard_Skills": {
        "Must_Have": {
            "Tech_Skills": [],
            "Certifications": []
        },
        "Nice_To_Have_Expansions": {
            "Tech_Siblings": []
        }
    },
    "Soft_Skills": []
}


############################################################
### 2. FILE: JD_BA_2_cleaned.json
############################################################
{
    "Metadata": {
        "Job_Title": "Product Manager",
        "Processed_At": "2026-04-12T00:26:30.154668"
    },
    "Hard_Constraints": {
        "M

# 🛠️ CHI TIẾT LOGIC XỬ LÝ TỐI ƯU (`process_jd_v2.py`)

Phiên bản này tập trung vào độ chính xác khi bóc tách vùng dữ liệu và chuẩn hóa thực thể dựa trên tri thức từ Taxonomy.

### 1. Phân tách ngữ nghĩa & Chunking (`semantic_slicing_and_chunking`)
- **Vá lỗi "Hố đen":** Sử dụng logic `first_must = min(must_pos) if must_pos else 0`. Nếu JD không có tiêu đề rõ ràng (như 'Essential Requirements'), hệ thống sẽ quét từ đầu file thay vì bỏ sót dữ liệu.
- **Phân vùng thông minh:** Tách văn bản thành `must_zone` (Yêu cầu) và `wish_zone` (Điểm cộng) dựa trên các bộ từ khóa Regex chuyên biệt.
- **Xử lý JD dài:** Mọi vùng văn bản vượt quá 1500 ký tự sẽ được chia nhỏ thành các `chunks` để tránh lỗi tràn context của mô hình GLiNER (giới hạn ~384-512 tokens).

### 2. Chiến thuật 2-Pass NER (Quét 2 lượt)
Để tránh việc các nhãn kỹ thuật (Hard Skills) lấn át các nhãn tâm lý/xã hội (Soft Skills), hệ thống thực hiện:
- **Lượt 1:** Trích xuất các thực thể cứng: `Job Role`, `Technical Skill`, `Tool`, `Certification`, `Education`, v.v.
- **Lượt 2:** Trích xuất riêng biệt nhãn `Soft Skill` trên cùng một đoạn văn bản.
- **Kết quả:** Tăng đáng kể tỷ lệ nhận diện các kỹ năng mềm ẩn trong văn bản JD phức tạp.

### 3. Chuẩn hóa & Hậu kiểm Taxonomy
- **Chuẩn hóa tra cứu:** Thay thế dấu gạch ngang bằng khoảng trắng (`replace("-", " ")`) giúp khớp chính xác các từ khóa như "Business-Analysis" với database.
- **Category Correction (Sửa lỗi danh mục):**
    - Nếu GLiNER nhận diện nhãn là `Certification` nhưng kết quả từ ChromaDB cho thấy thực thể đó nằm trong nhánh `Tools` hoặc `Technologies` của cây tri thức.
    - Hệ thống sẽ tự động chuyển thực thể đó sang rổ `Tech_Skills` (Ví dụ: Chuyển 'Android Auto' từ Chứng chỉ về đúng Kỹ thuật).

### 4. Expansion & Giới hạn nhiễu
- **Top-10 Expansion:** Khi lấy các kỹ năng liên quan (Siblings), hệ thống chỉ lấy 10 phần tử đầu tiên từ Taxonomy để đảm bảo danh sách `Nice_To_Have` không quá dài và loãng.
- **Logic Loại trừ (Deduplication):** Sử dụng `set.difference()` để đảm bảo:
    - Một kỹ năng đã là `Must_Have` thì không xuất hiện trong `Nice_To_Have`.
    - Kỹ năng bốc từ JD (`From_JD_Desirable`) được ưu tiên hơn kỹ năng gợi ý từ hệ thống (`From_Taxonomy_Expansion`).

In [ ]:
import re
import json
import os
import chromadb
from gliner import GLiNER
from pathlib import Path
from datetime import datetime

# --- KHỞI TẠO HỆ THỐNG ---
print(f"[DEBUG_LOG] Đang tải mô hình và kết nối database...")
DB_PATH = "/content/drive/MyDrive/Nlp resume ranker/chroma_db"
client = chromadb.PersistentClient(path=DB_PATH)

tech_col = client.get_collection(name="tech_ontology")
soft_col = client.get_collection(name="soft_skills_ontology")

# Sử dụng model đa ngôn ngữ v2.1
gliner_model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

JD_INPUT_DIR = "/content/drive/MyDrive/Nlp resume ranker/JD"
JD_OUTPUT_DIR = "/content/drive/MyDrive/Nlp resume ranker/Cleaned_JD_V2"
TECH_JSON_PATH = '/content/drive/MyDrive/Nlp resume ranker/taxonomy/taxonomy_processed/tech_ontology.json'

os.makedirs(JD_OUTPUT_DIR, exist_ok=True)

def load_raw_ontology(path):
    try:
        with open(path, 'r', encoding='utf-8') as f: return json.load(f)
    except: return {}

def get_by_path(root, path_str):
    keys = path_str.split('.')
    for k in keys:
        if isinstance(root, dict): root = root.get(k, {})
    return root if isinstance(root, list) else []

def semantic_slicing_and_chunking(text):
    """
    Phân tách văn bản thành Must_Zone và Wish_Zone dựa trên từ khóa.
    """
    MUST_REGEX = r'(?i)^(?:\s*|.*\n)(?:essential|requirements|required|must have|key responsibilities|qualifications|technical expertise)\b'
    WISH_REGEX = r'(?i)(desirable|pluses|nice to have|preferred|advantage|plus|good to have)'
    # STOP_REGEX chỉ khớp với tiêu đề độc lập ở đầu dòng
    STOP_REGEX = r'(?i)^\s*(?:benefits|what we offer|about company|closing date|equal opportunity)\b'

    # Tìm các vị trí mốc
    must_pos = [m.start() for m in re.finditer(MUST_REGEX, text)]
    wish_pos = [m.start() for m in re.finditer(WISH_REGEX, text)]
    stop_pos = [m.start() for m in re.finditer(STOP_REGEX, text, re.MULTILINE)]

    # [Vá lỗi Hố đen]: Nếu không tìm thấy tiêu đề, mặc định là 0
    first_must = min(must_pos) if must_pos else 0
    first_wish = min(wish_pos) if wish_pos else None
    first_stop = min(stop_pos) if stop_pos else len(text)

    must_zone = ""
    wish_zone = ""

    if first_wish is not None and first_wish < first_stop:
        must_zone = text[first_must:first_wish]
        wish_zone = text[first_wish:first_stop]
    else:
        must_zone = text[first_must:first_stop]

    def chunk_text(t, size=1500):
        return [t[i:i+size] for i in range(0, len(t), size)]

    return {
        "must_chunks": chunk_text(must_zone) if must_zone else [],
        "wish_chunks": chunk_text(wish_zone) if wish_zone else []
    }

def process_single_jd(jd_text, distance_threshold=0.4):
    results = {
        "Metadata": {"Job_Title": "Unknown", "Processed_At": datetime.now().isoformat()},
        "Hard_Constraints": {"Min_Experience_Years": None, "Required_Degree": "Not specified"},
        "Hard_Skills": {
            "Must_Have": {"Tech_Skills": set(), "Certifications": set()},
            "Nice_To_Have": {"From_JD_Desirable": set(), "From_Taxonomy_Expansion": set()}
        },
        "Soft_Skills": set()
    }

    exp_match = re.search(r'(\d+)\s*(?:year|năm)', jd_text, re.IGNORECASE)
    results["Hard_Constraints"]["Min_Experience_Years"] = int(exp_match.group(1)) if exp_match else 0

    zones = semantic_slicing_and_chunking(jd_text)
    # Lượt 1: Nhãn cứng | Lượt 2: Soft Skill
    pass_1_labels = ["Job Role", "Technical Skill", "Methodology", "Tool", "Certification", "Domain Knowledge", "Language", "Education"]
    pass_2_labels = ["Soft Skill"]
    raw_tech_tree = load_raw_ontology(TECH_JSON_PATH)

    def process_zone_chunks(chunks, zone_type):
        for chunk in chunks:
            # 2-Pass NER
            all_entities = gliner_model.predict_entities(chunk, pass_1_labels) + gliner_model.predict_entities(chunk, pass_2_labels)

            for ent in all_entities:
                label, text = ent['label'], ent['text'].strip()
                # [Chuẩn hóa tra cứu]
                clean_text = text.replace("-", " ").strip().lower()

                if label == "Job Role" and results["Metadata"]["Job_Title"] == "Unknown":
                    results["Metadata"]["Job_Title"] = text
                elif label == "Education" and results["Hard_Constraints"]["Required_Degree"] == "Not specified":
                    results["Hard_Constraints"]["Required_Degree"] = text

                elif label in ["Technical Skill", "Methodology", "Tool", "Certification", "Domain Knowledge", "Language"]:
                    query_res = tech_col.query(query_texts=[clean_text], n_results=1)
                    dist = query_res['distances'][0][0] if query_res['distances'] else 1.0
                    path = query_res['metadatas'][0][0]['path'] if dist < distance_threshold else ""

                    # [Category Correction]: Nếu NER bảo là Cert nhưng Taxonomy bảo là Tool/Tech
                    final_label = label
                    if label == "Certification" and dist < distance_threshold:
                        if any(x in path.lower() for x in ["tool", "technology", "software"]):
                            final_label = "Technical Skill"

                    target_bucket = "Certifications" if final_label == "Certification" else "Tech_Skills"
                    matched_skill = query_res['documents'][0][0] if dist < distance_threshold else text

                    if zone_type == "must":
                        results["Hard_Skills"]["Must_Have"][target_bucket].add(matched_skill)
                        # [Giới hạn Expansion - Top 10]
                        if target_bucket == "Tech_Skills" and dist < distance_threshold:
                            neighbors = get_by_path(raw_tech_tree, path)[:10]
                            for n in neighbors:
                                if isinstance(n, str): results["Hard_Skills"]["Nice_To_Have"]["From_Taxonomy_Expansion"].add(n.lower())
                    else:
                        results["Hard_Skills"]["Nice_To_Have"]["From_JD_Desirable"].add(matched_skill)

                elif label == "Soft Skill":
                    query_res = soft_col.query(query_texts=[clean_text], n_results=1)
                    dist = query_res['distances'][0][0] if query_res['distances'] else 1.0
                    results["Soft_Skills"].add(query_res['documents'][0][0] if dist < distance_threshold else text)

    process_zone_chunks(zones["must_chunks"], "must")
    process_zone_chunks(zones["wish_chunks"], "wish")

    must_tech = results["Hard_Skills"]["Must_Have"]["Tech_Skills"]
    results["Hard_Skills"]["Nice_To_Have"]["From_JD_Desirable"] = list(results["Hard_Skills"]["Nice_To_Have"]["From_JD_Desirable"].difference(must_tech))
    results["Hard_Skills"]["Nice_To_Have"]["From_Taxonomy_Expansion"] = list(results["Hard_Skills"]["Nice_To_Have"]["From_Taxonomy_Expansion"].difference(must_tech).difference(set(results["Hard_Skills"]["Nice_To_Have"]["From_JD_Desirable"])))

    results["Hard_Skills"]["Must_Have"]["Tech_Skills"] = list(must_tech)
    results["Hard_Skills"]["Must_Have"]["Certifications"] = list(results["Hard_Skills"]["Must_Have"]["Certifications"])
    results["Soft_Skills"] = list(results["Soft_Skills"])

    return results

def batch_process_jds():
    print(f"\n{'='*20} BẮT ĐẦU XỬ LÝ TỐI ƯU {'='*20}")
    jd_files = [f for f in os.listdir(JD_INPUT_DIR) if os.path.isfile(os.path.join(JD_INPUT_DIR, f))]
    for filename in jd_files:
        try:
            with open(os.path.join(JD_INPUT_DIR, filename), 'r', encoding='utf-8') as f: jd_content = f.read()
            enriched_data = process_single_jd(jd_content)
            with open(os.path.join(JD_OUTPUT_DIR, Path(filename).stem + "_cleaned.json"), 'w', encoding='utf-8') as f:
                json.dump(enriched_data, f, indent=4, ensure_ascii=False)
            print(f"  [OK] Processed: {filename}")
        except Exception as e: print(f"  [ERROR] {filename}: {str(e)}")

if __name__ == "__main__":
    batch_process_jds()

[DEBUG_LOG] Đang tải mô hình và kết nối database...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


==================== BẮT ĐẦU XỬ LÝ TỐI ƯU ====================
  [OK] Processed: JD_BA_1.txt
  [OK] Processed: JD_BA_2.txt
  [OK] Processed: JD_SW_1.txt
  [OK] Processed: JD_SW_2.txt
  [OK] Processed: JD_SW_3.txt
  [OK] Processed: JD_SW_5.txt
  [OK] Processed: JD_SW_6.txt
  [OK] Processed: JD_PM_2.txt
  [OK] Processed: JD_SM_2.txt
  [OK] Processed: JD_DA_DE_1.txt
  [OK] Processed: JD_DA_DE_2.txt
  [OK] Processed: JD_DA_DE_3.txt
  [OK] Processed: JD_BA_3.txt
  [OK] Processed: JD_BA_4.txt
  [OK] Processed: JD_PM_1.txt
  [OK] Processed: JD_SM_1.txt
  [OK] Processed: JD_SW_4.txt


## 📊 KIỂM TRA KẾT QUẢ ĐẦU RA (process_jd_v2.py)

Sử dụng đoạn code dưới đây để xem nhanh các file JSON đã được xử lý trong thư mục `Cleaned_JD_V2`.

In [ ]:
import os
import json

# Đường dẫn tới thư mục chứa JD đã xử lý
CLEANED_DIR = "/content/drive/MyDrive/Nlp resume ranker/Cleaned_JD_V2"

if not os.path.exists(CLEANED_DIR):
    print(f"[!] Thư mục {CLEANED_DIR} không tồn tại.")
else:
    cleaned_files = [f for f in os.listdir(CLEANED_DIR) if f.endswith('.json')]

    if not cleaned_files:
        print("[!] Không tìm thấy file JSON nào trong thư mục đầu ra.")
    else:
        print(f"[INFO] Đang hiển thị nội dung của {len(cleaned_files)} file đã xử lý theo SCHEMA MỚI:\n")

        for i, filename in enumerate(cleaned_files, 1):
            file_path = os.path.join(CLEANED_DIR, filename)

            print(f"{'#'*60}")
            print(f"### {i}. FILE: {filename}")
            print(f"{'#'*60}")

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    # In định dạng JSON đẹp mắt
                    print(json.dumps(data, indent=4, ensure_ascii=False))
            except Exception as e:
                print(f"[ERROR] Không thể đọc file {filename}: {e}")

            print("\n")

[INFO] Đang hiển thị nội dung của 17 file đã xử lý theo SCHEMA MỚI:

############################################################
### 1. FILE: JD_BA_1_cleaned.json
############################################################
{
    "Metadata": {
        "Job_Title": "OPTiX Project Director",
        "Processed_At": "2026-04-12T00:36:55.802294"
    },
    "Hard_Constraints": {
        "Min_Experience_Years": 0,
        "Required_Degree": "Not specified"
    },
    "Hard_Skills": {
        "Must_Have": {
            "Tech_Skills": [
                "waterfall",
                "BA methodologies",
                "Specialist/Technical Expertise",
                "business analysis toolsets",
                "uml"
            ],
            "Certifications": []
        },
        "Nice_To_Have": {
            "From_JD_Desirable": [
                "IIBA certifications",
                "BCS International Diploma in Business Analysis"
            ],
            "From_Taxonomy_Expansion": [
 